# Phase 2: G2P fine-tuning (ByT5-small) - Kaggle version

Same training as `notebooks/g2p_finetune.ipynb`, adapted for Kaggle Notebooks instead of Colab (different free GPU quota: 30 hrs/week on Kaggle vs. Colab's opaque daily-ish allowance).

**Setup on kaggle.com:**
1. Create a new Notebook.
2. Settings (right panel) > Accelerator > **GPU T4 x2** (or P100).
3. Right panel > Add Input > **Upload** tab > drag in `train.csv`, `val.csv`, `test.csv` from `data/processed/g2p/` (built locally via `python scripts/build_g2p_dataset.py`). This creates a private dataset and mounts it read-only under `/kaggle/input/<something>/` - the exact folder name varies, so the first code cell below auto-detects it rather than hardcoding a path.

The **test** split is exactly the same 500 items already scored against raw TTS output in Phase 1 (`data/results/benchmark_results.csv`), so the PER this notebook reports is directly comparable to that.

In [ ]:
!pip install -q -U transformers datasets accelerate sentencepiece

In [ ]:
import glob
import os

candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "train.csv not found under /kaggle/input/ - use 'Add Input > Upload' in the "
        "right panel to attach train.csv, val.csv, and test.csv first."
    )
DATA_DIR = os.path.dirname(candidates[0])
print(f"Found data at: {DATA_DIR}")
print(os.listdir(DATA_DIR))

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

def load_split(name):
    # keep_default_na=False: some real WikiPron words (e.g. Vietnamese "nan")
    # collide with pandas' default NA sentinel strings ("nan", "NA", "null",
    # "None", ...) and would otherwise silently become missing values here,
    # breaking tokenization downstream with a cryptic NoneType error.
    df = pd.read_csv(f"{DATA_DIR}/{name}.csv", keep_default_na=False, na_values=[])
    df["input_text"] = "[" + df["lang_id"] + "] " + df["word"]
    df["target_text"] = df["ipa"]
    return Dataset.from_pandas(
        df[["input_text", "target_text", "lang_id", "word", "ipa"]], preserve_index=False
    )

raw_datasets = DatasetDict({
    "train": load_split("train"),
    "val": load_split("val"),
    "test": load_split("test"),
})
raw_datasets

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

MAX_INPUT_LEN = 64
MAX_TARGET_LEN = 64

def preprocess(batch):
    model_inputs = tokenizer(batch["input_text"], max_length=MAX_INPUT_LEN, truncation=True)
    labels = tokenizer(text_target=batch["target_text"], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = raw_datasets.map(preprocess, batched=True, remove_columns=["input_text", "target_text"])

In [ ]:
import torch
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# T5/ByT5 models are well documented to be numerically unstable under fp16
# (their internals overflow fp16's range, so the gradient scaler silently
# skips every optimizer step to avoid NaN weights - training looks like
# it's running but the model never actually updates: exactly the
# "training_loss=0.0, eval_loss=nan" symptom). Use bf16 only if the GPU
# actually supports it (T4 doesn't; P100/A100 support varies); otherwise
# plain fp32.
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/g2p_byt5_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=False,
    bf16=use_bf16,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["val"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

## Save checkpoint

No Drive mount needed on Kaggle - anything written to `/kaggle/working/` shows up in the notebook's **Output** tab after you save/commit, and can be downloaded or reused from there directly.

In [ ]:
SAVE_DIR = "/kaggle/working/g2p_byt5_small"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")

## Run inference on the held-out test set

In [ ]:
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def predict(word, lang_id, max_length=MAX_TARGET_LEN):
    input_text = f"[{lang_id}] {word}"
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=max_length)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

test_df = raw_datasets["test"].to_pandas()
test_df["predicted_ipa"] = [predict(w, l) for w, l in zip(test_df["word"], test_df["lang_id"])]
test_df.rename(columns={"ipa": "reference_ipa"}, inplace=True)
test_df[["lang_id", "word", "reference_ipa", "predicted_ipa"]].head()

## Score predictions

Inlined port of `src/pronunciation_benchmark/scoring/per.py` + `scoring/normalize.py` so this notebook is self-contained (no package install needed - this repo has no GitHub remote yet). `align` below is simplified to just the edit-distance count (no S/D/I breakdown) since that's all corpus PER needs - keep in sync with the source files if that scoring logic changes.

In [ ]:
# --- mirrors src/pronunciation_benchmark/scoring/per.py and scoring/normalize.py ---

def tokenize_ipa(ipa):
    return ipa.split()

_TONE_TOKEN_CHARS = set("˥˦˧˨˩ˀ")
_TONE_ACCENT_MARKS = "̀́̄"

def strip_suprasegmentals(tokens):
    result = []
    for tok in tokens:
        if tok and all(ch in _TONE_TOKEN_CHARS for ch in tok):
            continue
        stripped = "".join(ch for ch in tok if ch not in _TONE_ACCENT_MARKS)
        if stripped:
            result.append(stripped)
    return result

def align(reference, hypothesis):
    """Returns (edit_distance, len(reference))."""
    n, m = len(reference), len(hypothesis)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if reference[i - 1] == hypothesis[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[n][m], n

def phoneme_error_rate(reference, hypothesis):
    edits, ref_len = align(reference, hypothesis)
    if ref_len == 0:
        return 0.0 if edits == 0 else float("inf")
    return edits / ref_len

def corpus_phoneme_error_rate(pairs):
    total_edits = total_ref_len = 0
    for reference, hypothesis in pairs:
        edits, ref_len = align(reference, hypothesis)
        total_edits += edits
        total_ref_len += ref_len
    return total_edits / total_ref_len if total_ref_len else 0.0

In [ ]:
test_df["ref_tokens"] = test_df["reference_ipa"].apply(lambda s: strip_suprasegmentals(tokenize_ipa(s)))
test_df["hyp_tokens"] = test_df["predicted_ipa"].apply(lambda s: strip_suprasegmentals(tokenize_ipa(s)))
test_df["per"] = [phoneme_error_rate(r, h) for r, h in zip(test_df["ref_tokens"], test_df["hyp_tokens"])]

print("G2P model corpus PER by language:")
for lang_id, group in test_df.groupby("lang_id"):
    pairs = list(zip(group["ref_tokens"], group["hyp_tokens"]))
    print(f"  {lang_id}: {corpus_phoneme_error_rate(pairs):.3f}  (n={len(group)})")

## Export predictions for the local before/after comparison

Writes `g2p_predictions.csv` to `/kaggle/working/` - after this cell runs, use Kaggle's **Save Version** (top right) to commit the notebook, then download `g2p_predictions.csv` from the Output tab. Move it to `data/results/g2p_predictions.csv` in the repo, then run `python scripts/compare_g2p_vs_tts.py` locally.

In [ ]:
out_df = test_df[["lang_id", "word", "reference_ipa", "predicted_ipa"]]
out_path = "/kaggle/working/g2p_predictions.csv"
out_df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Wrote {len(out_df)} predictions -> {out_path}")
print("Save Version (top right) to commit, then download g2p_predictions.csv from the Output tab.")